# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nnanwubeikenna-prog/ikenna-flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Binary Classification & Risk Ranking  
**Why:** The goal is to predict whether an existing, high-visibility content URL will experience severe performance decay in the upcoming 30-day window (`is_declining = 1` vs `0`), and rank flagged pages by predicted risk probability so content editors can prioritize which pages to audit and refresh.

In [1]:
import os, subprocess, sys
import numpy as np
import pandas as pd

# Setup repo access if in Colab
if 'google.colab' in sys.modules and not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.exists('flyrank-ml-internship-starter'):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git"])
    if os.path.exists('flyrank-ml-internship-starter/data'):
        df = pd.read_csv('flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv')
    else:
        df = pd.read_csv('data/raw/content_refresh_anonymized.csv' if os.path.exists('data/raw/content_refresh_anonymized.csv') else '../data/raw/content_refresh_anonymized.csv')

df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
print(f"Total dataset rows: {len(df)}")
print("Target label distribution (1 = down / decaying, 0 = stable/up):")
print(df['is_declining_label'].value_counts(normalize=True).round(3))

Total dataset rows: 30000
Target label distribution (1 = down / decaying, 0 = stable/up):
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label` (Binary: 1 if `trend_direction == 'down'`, else 0).  
**Source:** This is an observed historical outcome derived from whether a page's impressions dropped by more than 20% month-over-month. We strictly avoid using `trend_pct` directly as a training feature to prevent data leakage.

In [2]:
down_pages = df[df['is_declining_label'] == 1]
print("Distribution of trend_pct for down-trending pages:")
print(down_pages['trend_pct'].describe().round(3))

Distribution of trend_pct for down-trending pages:
count    16262.000
mean       -58.114
std         23.489
min       -100.000
25%        -75.900
50%        -55.600
75%        -38.500
max        -20.000
Name: trend_pct, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric:** Precision@50 (and Precision@20).  
**Why:** Content teams have limited operational bandwidth (typically 20 to 50 articles per sprint). If an ML model recommends 50 pages for urgent rewrite, we need at least 60%+ of those recommendations to be genuine decay cases (minimizing false positives and avoiding wasted editorial hours).

In [3]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return float(np.mean(topk))

np.random.seed(42)
random_scores = np.random.rand(len(df))
print(f"Random Baseline Precision@50: {precision_at_k(random_scores, df['is_declining_label'].values, k=50):.3f}")

Random Baseline Precision@50: 0.560


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** Exactly **one content URL (web page) over a trailing 90-day observation window**.  
Each row aggregates historical signals (impressions, clicks, CTR, position, staleness) without exposing future performance.

In [4]:
unit_cols = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'is_declining_label']
print("Unit of analysis (1 row = 1 page):")
display(df[unit_cols].head(5))

Unit of analysis (1 row = 1 page):


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,is_declining_label
0,187,20,3803,10.6,0.76,1
1,445,25,15320,20.3,0.05,1
2,141,20,12581,36.5,0.09,1
3,463,22,11751,6.2,0.49,0
4,263,14,19140,44.0,0.13,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a rule:** A static rule (such as `days_since_last_update > 180`) fails because evergreen articles can remain stable for years, while pages in volatile topics decay in weeks. ML combines non-linear interactions across position drift, impressions, and engagement to separate true decay from normal traffic variance.

In [5]:
# Evaluate hand rule: stale and visible
hand_rule_score = ((df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)).astype(int) * df['impressions_90d']
print(f"Hand Rule Precision@50: {precision_at_k(hand_rule_score.values, df['is_declining_label'].values, k=50):.3f}")

Hand Rule Precision@50: 0.680


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.